In [1]:
# -*- coding: utf-8 -*-
"""
Created on Wed Oct 21 14:12:42 2020

@author: clara, viggy
"""

import numpy as np
import matplotlib.pyplot as plt
import copy
import statistics
import pandas as pd
import os
import re
import glob
from func_stepFrequency import get_gait, get_peaks, calc_step_width, calculate_step_stride_length, calc_interstep_all, get_step_lengths, get_frames, select_valid_lanes, variability, asymmetry, intersect, calculate_stance_swing, calculate_percent_cycle, double_support_time, apply_lowpass_filter
import traceback
import unicodedata


In [2]:
new_frames = pd.DataFrame()
error_subj = []
error_trace = []
error_idx = []

In [3]:
INDICES = ["trial_num", "test_type",
                      "Avg_LF_stance_time", 
                      "Avg_LF_swing_time",
                      "Avg_RF_stance_time",
                      "Avg_RF_swing_time",
                      "Variability_LF_stance_time",
                      "Variability_LF_swing_time",
                      "Variability_RF_stance_time",
                      "Variability_RF_swing_time",
                      "Asymmetry_Stance_time",
                      "Asymmetry_Swings_time",
                      "total_%_swing_time",
                      "total_%_stance_time",
                      "double_support_time",
                      "Avg_InterStepTime",
                      "Variability_InterStepTime",
                      "Frequency",
                      "Variance_COM_y",
                      "speed",
                      "Avg_stride_length",
                      "Variance_stride_length",
                      "Variability_stride_length",
                      "Asymmetry_stride_length",
                      "Avg_step_length",
                      "Variance_step_length",
                      "Variability_step_length",
                      "Avg_step_width",
                      "Variance_step_width",
                      "Variability_step_width",
                      "Avg_LF_max_height",
                      "Avg_RF_max_height",
                      "Variability_LF_max_height",
                      "Variability_RF_max_height",
                      "Asymmetry_F_max_height"]

## Load COM Data

In [ ]:
#filelocation = glob.glob(f'../data/gang/*.txt')
#filelocation = glob.glob(f'Patient_txt_PD_reup/*.txt')
#filelocation = glob.glob(f'Patient_txt_ataxia_reup/*.txt')
#filelocation = glob.glob(f'missing_txt_files_20250818/*.txt')
filelocation = glob.glob(f'all_participants_061826/*.txt')


all_results = pd.DataFrame()
counter = 0
PATTERN = r'-[2-9]-'

for files in filelocation:
    try:
        data = np.loadtxt(files, encoding = 'latin1') 
        #sub_name = [s[0:6] for s in os.path.basename(files).split()]
        file_strip = os.path.basename(files)
        sub_name = re.match(r'\d+', file_strip).group()  # leading digits: 101-001->101, 215147-001->215147 (handles 3- and 6-digit IDs)
        
        #search for multiple trials
        search = re.search(PATTERN, file_strip)
        if search != None: #match
            trial_num = search.group(0)[1]
        else: #no match
            trial_num = "1"
    
        subject = sub_name
        
        '''
        print("file_strip", file_strip)
        print("sub_name", sub_name)
        print("trial_num", trial_num)
        
        print(f'Starting analysis for subject: {subject}')
        print(f"---------index {counter}---------")
        '''
        
        # NFC-normalize: macOS filenames are NFD, so a precomposed literal ('rückwärtsgang')
        # won't substring-match unless we normalize both to the same form.
        files_norm = unicodedata.normalize('NFC', files).lower()
        if 'normalergang' in files_norm:
            test_type = 'NG'
            #sub_name += num
            #sub_name += "_NG"
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4', 'lane_5', 'lane_6']
        elif 'rückwärtsgang' in files_norm:
            test_type = 'RG'
            #sub_name += num
            #sub_name += '_RG'
            #lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4', 'lane_5', 'lane_6']
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4']
        elif 'tandemgang' in files_norm:
            test_type = 'TG'
            #sub_name += num
            #sub_name += '_TG'
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4']

        timeframes = data[:,1] #fractions of a second?
        time = timeframes / 60 #seconds or minutes?
        
        COM_x = data [:,2] #sagittal / anterior posterior
        COM_y = data[:,3] # right left
        COM_z = data[:,4] # vertical / up down
        # Subject 101 was exported with an extra Neck segment (38 cols vs the usual 32),
        # which shifts every position block after it by +3, so the feet land 3 columns later.
        # COM/Pelvis (first position block, cols 2-4) is unaffected. Detect by column count.
        foot_off = 3 if data.shape[1] == 38 else 0
        LeftFoot_x = data [:,11+foot_off]
        LeftFoot_y = data [:,12+foot_off]
        LeftFoot_z = data [:,13+foot_off]
        RightFoot_x = data [:,14+foot_off] #front-back in m
        RightFoot_y = data [:,15+foot_off] #left-right
        RightFoot_z = data [:,16+foot_off] #height

        # ---- Butterworth low-pass at construction (4th-order, 6 Hz, zero-phase) ----
        # Filter all position channels once here so every downstream consumer (get_peaks, the
        # stance/swing + DST path, and the x/y distance features: stride/step length, step
        # width, speed, COM_y variance) sees the same filtered signal. get_peaks runs with
        # filt=False (its default) to avoid double-filtering. Note: COM_x feeds gait-plate/lane
        # detection below, so that now also runs on filtered data (negligible on a slow path).
        COM_x = apply_lowpass_filter(COM_x)
        COM_y = apply_lowpass_filter(COM_y)
        COM_z = apply_lowpass_filter(COM_z)
        LeftFoot_x = apply_lowpass_filter(LeftFoot_x)
        LeftFoot_y = apply_lowpass_filter(LeftFoot_y)
        LeftFoot_z = apply_lowpass_filter(LeftFoot_z)
        RightFoot_x = apply_lowpass_filter(RightFoot_x)
        RightFoot_y = apply_lowpass_filter(RightFoot_y)
        RightFoot_z = apply_lowpass_filter(RightFoot_z)

        COM_z_time = np.array([time]+[COM_z]) #COM_z values with corresponding time
        COM_x_time = np.array([time]+[COM_x])
        COM_y_time = np.array([time]+[COM_y])

        
        ## Calculate Time at Gait Plate
        maxCOMx = max(COM_x)-1
        minCOMx = min(COM_x)+1


        COM_x_new = copy.copy(COM_x)
        COM_x_new[COM_x_new > maxCOMx] = 9.999
        COM_x_new[COM_x_new < minCOMx] = 9.999
        COM_x_new_time = np.array([time] + [COM_x_new])
        
        _, lane_frames = get_frames(COM_x_new_time, sub_name, test_type)

        # get_frames over-segments some recordings into short junk fragments (zero foot-minima)
        # that would land in the first-N selection and crash the step/stance code. Pick the first
        # len(lanes) detected lanes that are real walking passes (>=1 foot-minimum in BOTH feet),
        # skipping the junk. For non-degenerate files this is identical to lane_frames[:len(lanes)].
        selected_frames = select_valid_lanes(lane_frames, len(lanes), LeftFoot_z, RightFoot_z)
        # allow files with fewer valid passes than requested: process whatever
        # we actually have (e.g. short recordings) instead of crashing on missing lanes.
        lanes = lanes[:len(selected_frames)]
    
        com_data = {}
        lf_data = {}
        rf_data = {} 

        ## Build COM data dict per lane
        for lane in range(len(lanes)):
            lane_name = "lane_" + str(lane+1)
            x_data = COM_x_new_time[:, selected_frames[lane][0]:selected_frames[lane][1]]
            y_data = COM_y_time[:, selected_frames[lane][0]:selected_frames[lane][1]]
            z_data = COM_z_time[:, selected_frames[lane][0]:selected_frames[lane][1]]
            com_data[lane_name] = [x_data, y_data, z_data]

        ## Build Left/Right Foot data dict per lane
        for lane in range(len(lanes)):
            lane_name = "lane_" + str(lane+1)
            x_data = LeftFoot_x[selected_frames[lane][0]:selected_frames[lane][1]]
            y_data = LeftFoot_y[selected_frames[lane][0]:selected_frames[lane][1]]
            z_data = LeftFoot_z[selected_frames[lane][0]:selected_frames[lane][1]]
            lf_data[lane_name] = [x_data, y_data, z_data]

            x_data = RightFoot_x[selected_frames[lane][0]:selected_frames[lane][1]]
            y_data = RightFoot_y[selected_frames[lane][0]:selected_frames[lane][1]]
            z_data = RightFoot_z[selected_frames[lane][0]:selected_frames[lane][1]]
            rf_data[lane_name] = [x_data, y_data, z_data]

        
        var_all = []
        v_all = []

        Stride_length_all = [] #actually stride lengths
        Stride_length_all_L = [] 
        Stride_length_all_R = [] 

        step_width_all = []
        LF_height_sum = []
        RF_height_sum = []
        step_lengths = [] # step_lengths

        #swing & stances
        all_stances_L = []
        all_stances_R = []
        all_swings_L = []
        all_swings_R = []

        #percent swing and stances, currently calculated with all
        all_pc_st = []
        all_pc_sw = []
        
        #double support times for each lane in file
        dst_list = []

        for lane in lanes:
            #print(f'-------------------------{lane}-------------------------------')
            COM_xy_Lane_tp_new,  LF_xy_Lanetp_new, RF_xy_Lanetp_new = get_gait(lane, com_data, lf_data, rf_data, plot='off')

            var_lane = statistics.variance(COM_xy_Lane_tp_new[0,:])
            var_all.append(var_lane)
            
            v_lane = abs((COM_xy_Lane_tp_new[1,-1]-COM_xy_Lane_tp_new[1,0])/(com_data[lane][0][0,-1]-com_data[lane][0][0,0])) #(pos_end - pos_start) / (time_end - time_start)
            
            v_all.append(v_lane) #average of this is the speed
            
            peaks_COM, peaks_left, peaks_right, peaks_left_corrected, peaks_right_corrected, peaks_timeonly, peaks_heightonly, minima_RF, minima_LF = get_peaks(lane, com_data, lf_data, rf_data, plot = 'off') ##
            
            #STRIDE LENGTH
            stride_length, stride_length_L, stride_length_R = calculate_step_stride_length(lane, com_data, lf_data, rf_data)
            Stride_length_all += list(stride_length) 
            Stride_length_all_L += list(stride_length_L)
            Stride_length_all_R += list(stride_length_R)
            
            #STEP LENGTH
            step_lengths += list(get_step_lengths(minima_LF, minima_RF, LF_xy_Lanetp_new, RF_xy_Lanetp_new, lane)) #--> STEPS
            
            #STEP WIDTH
            step_width_all.append(calc_step_width(lane, com_data, lf_data, rf_data))

            LF_height_sum += list(peaks_left_corrected)
            RF_height_sum += list(peaks_right_corrected)
            
            #SWING AND STANCE TIME
            R, L, _, _ = intersect(lane, rf_data, lf_data, minima_RF, minima_LF, plot="off") ##
            swing_R, stance_R, swing_L, stance_L, dst_base = calculate_stance_swing(lane, L, R, lf_data, rf_data, com_data)
            dst = double_support_time(dst_base)
            pc_sw, pc_st = calculate_percent_cycle(swing_R, stance_R, swing_L, stance_L)
            
            dst_list.append(dst)
            
            all_stances_L += list(stance_L)
            all_stances_R += list(stance_R)
            all_swings_L += list(swing_L)
            all_swings_R += list(swing_R)
            
            all_pc_st.append(pc_st)
            all_pc_sw.append(pc_sw)
            
            plt.close()
            
        ## STANCE/SWING ##

        all_stances_L = np.array(all_stances_L).flatten() #flatten to 1d
        all_stances_R = np.array(all_stances_R).flatten()
        all_swings_L = np.array(all_swings_L).flatten()
        all_swings_R = np.array(all_swings_R).flatten()

        # average swing & stance, each leg (4)
        stance_L_avg = np.mean(all_stances_L)
        stance_R_avg = np.mean(all_stances_R)
        swing_L_avg = np.mean(all_swings_L)
        swing_R_avg = np.mean(all_swings_R)

        # average % swing and % stance 
        pc_st_avg = np.mean(all_pc_st)
        pc_sw_avg = np.mean(all_pc_sw)

        # TOTAL stance and swing percentages --> effectively the same
        total = np.sum(all_stances_L) + np.sum(all_stances_R) + np.sum(all_swings_L) + np.sum(all_swings_R)
        total_pc_sw = (np.sum(all_swings_L) + np.sum(all_swings_R)) / total
        total_pc_st = (np.sum(all_stances_L) + np.sum(all_stances_R)) / total

        # variability, both legs (1)
        stances_L_variability = variability(all_stances_L)
        stances_R_variability = variability(all_stances_R)
        swings_L_variability = variability(all_swings_L)
        swings_R_variability = variability(all_swings_R)

        # asymmetry, between legs (1)
        stances_asymmetry = asymmetry(all_stances_L, all_stances_R)
        swings_asymmetry = asymmetry(all_swings_L, all_swings_R)
        
        #double support time
        dst_avg = np.mean(dst_list)

        #interstep & frequency
        InterStepTime_avg, InterStepTime_arr = calc_interstep_all(lanes, com_data, lf_data, rf_data)
        Variability_InterStepTime = variability(InterStepTime_arr)
        Frequency = 1/InterStepTime_avg

        #### Calculate Variance of COMy and Velocity/Speed

        ## Durchschnittsgeschwindigkeit:
        variance_COMy = np.mean(var_all)
        speed = np.mean(v_all)

        #STRIDE
        stride_length_avg = np.mean(Stride_length_all) # Average STRIDE Length, all step lengths from right and left foot...
        stride_length_var = statistics.variance(Stride_length_all) # Variance
        stride_length_variability = variability(Stride_length_all)
        stride_length_asymmetry = asymmetry(Stride_length_all_L, Stride_length_all_R)

        #STEP -- NEW
        ac_step_length_avg = np.mean(step_lengths)
        ac_step_length_variance = statistics.variance(step_lengths)
        ac_step_length_variability = variability(step_lengths)

        step_width_all = [item for sublist in step_width_all for item in sublist] # macht aus einem nested array ein 1D-array
        #print(step_width_all)
        ##plt.plot(step_width_all, 'x')
        step_width_avg = np.mean(step_width_all)
        var_step_width = statistics.variance(step_width_all)
        step_width_variability = variability(step_width_all)

        #### Calculate Foot Height at Maximum Elevation
        LF_height_avg = np.mean(LF_height_sum)
        RF_height_avg = np.mean(RF_height_sum)

        foot_height_asymmetry = asymmetry(LF_height_sum, RF_height_sum)
        LF_height_variability = variability(LF_height_sum)
        RF_height_variability = variability(RF_height_sum)
        
        df = pd.DataFrame([trial_num,
                    test_type, 
                   stance_L_avg,
                   swing_L_avg,
                   stance_R_avg,
                   swing_R_avg,
                   stances_L_variability,
                   swings_L_variability,
                   stances_R_variability,
                   swings_R_variability,
                   stances_asymmetry,
                   swings_asymmetry,
                   total_pc_sw,
                   total_pc_st,
                   dst_avg,
                   InterStepTime_avg,
                   Variability_InterStepTime,
                   Frequency,
                   variance_COMy,
                   speed,
                   stride_length_avg,
                   stride_length_var,
                   stride_length_variability,
                   stride_length_asymmetry,
                   ac_step_length_avg,
                   ac_step_length_variance,
                   ac_step_length_variability,
                   step_width_avg,
                   var_step_width,
                   step_width_variability,
                   LF_height_avg,
                   RF_height_avg,
                   LF_height_variability,
                   RF_height_variability,
                   foot_height_asymmetry], index= INDICES, columns = [sub_name])

        all_results = pd.concat([all_results, df.T], axis = 0)
        all_results.to_csv('all_participants_062626.csv')

        #df.to_csv(f'../export/{sub_name}.csv')
        #df.to_csv(f'out/{sub_name}.csv') 
        #print(f'Analysis done for subject: {sub_name}')
        #plt.close()
        counter += 1
        
    except Exception as e:
        print("file_strip", file_strip)
        print("sub_name", sub_name)
        print("trial_num", trial_num)
        
        print(f'Starting analysis for subject: {subject}')
        print(f"---------index {counter}---------")
        print(f'Exception occured: {e}')
        print(traceback.format_exc())
        error_subj.append(str(sub_name))
        error_trace.append(traceback.format_exc())
        error_idx.append(counter)
        counter += 1
        
        #printing plots only for debugging
        print("trying debugging plots")
        try:
            for lane in lanes:
                _ = get_peaks(lane, com_data, lf_data, rf_data, plot = 'off') ##
                _ = intersect(lane, rf_data, lf_data, minima_RF, minima_LF, plot = 'off') ##
        except Exception as e2:
            print(f'Exception occured: {e}')
            print(traceback.format_exc())

       
       
        

#print(results)
#results.to_csv(f'../export/results_walks.csv')
#results.to_excel(f'../export/results_walks.xlsx')

file_strip 113-2-003-rückwärtsgang_Seg_RightFoot_LeftFoot_RightForeArm_LeftForeArm_Pelvis_Data_position_acceleration.txt
sub_name 113
trial_num 2
Starting analysis for subject: 113
---------index 0---------
Exception occured: name 'test_type' is not defined
Traceback (most recent call last):
  File "/var/folders/nm/1x6qm5g522n2n1mmwkfs5m8h0000gn/T/ipykernel_24206/2846275178.py", line 102, in <module>
    _, lane_frames = get_frames(COM_x_new_time, sub_name, test_type)
NameError: name 'test_type' is not defined

trying debugging plots
Exception occured: name 'test_type' is not defined
Traceback (most recent call last):
  File "/var/folders/nm/1x6qm5g522n2n1mmwkfs5m8h0000gn/T/ipykernel_24206/2846275178.py", line 102, in <module>
    _, lane_frames = get_frames(COM_x_new_time, sub_name, test_type)
NameError: name 'test_type' is not defined

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/var/folders/nm/1x6qm5g522n2n1mmwkfs

In [ ]:
for i in all_results["double_support_time"]:
    print(i)

0.0404021644566128


In [ ]:
all_results.to_csv('out/all_results_dst.csv')


In [ ]:
len(error_subj)

28

In [ ]:
demos = pd.read_csv("Demographics_PD.csv")

In [ ]:
#subset to get rid of
demos_sub = demos.iloc[0:28, ]

demos_sub["ID"] = demos_sub["ID"].astype(int)
demos_sub["ID"] = demos_sub["ID"].astype(str)

#replace NV values with as control
demos_sub["Control"][14:16] = pd.Series(["1","1"])

#create mapping from control to subject ID
map = demos_sub[["ID", "Control"]]
map.set_index('ID', inplace=True)

mapping = map["Control"].to_dict()


demos_sub["Control"] = demos_sub["ID"].map(mapping)
all_results["ID"] = all_results.index
all_results["Control"] = all_results["ID"].map(mapping) #create Control column

/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demos_sub["ID"] = demos_sub["ID"].astype(int)
/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demos_sub["ID"] = demos_sub["ID"].astype(str)
/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a D

In [ ]:
all_results.to_csv("out/all_results_dst_ctrl.csv")